# ENSO Forecasting: OLS Linear Regression Baseline

This notebook trains an Ordinary Least Squares Linear Regression baseline to predict the next-month ENSO sea surface temperature from a 12-month sliding window.

The original chronological train/test split is preserved, and the final test set is used only for model evaluation.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load the Data

In [ ]:
# This works whether the notebook is run from the notebooks folder or the project root.
data_dir = Path("../data")
if not data_dir.exists():
    data_dir = Path("data")

X_train = pd.read_csv(data_dir / "X_train.csv")
X_test = pd.read_csv(data_dir / "X_test.csv")
y_train = pd.read_csv(data_dir / "y_train.csv").to_numpy(dtype=np.float32).ravel()
y_test = pd.read_csv(data_dir / "y_test.csv").to_numpy(dtype=np.float32).ravel()

print("Loaded data from:", data_dir.resolve())
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 3. Prepare Data for Model

In [ ]:
total_samples = len(X_train) + len(X_test)
train_pct = len(X_train) / total_samples * 100
test_pct = len(X_test) / total_samples * 100

print(f"Total samples: {total_samples}")
print(f"Train samples: {len(X_train)} ({train_pct:.1f}%)")
print(f"Test samples: {len(X_test)} ({test_pct:.1f}%)")
print(f"Features: {list(X_train.columns)}")
print("Target: target_t")
print("Split method: chronological, no shuffle")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

all_y = np.concatenate([y_train, y_test])
axes[0].plot(range(len(y_train)), y_train, label="Train", color="steelblue")
axes[0].plot(range(len(y_train), len(all_y)), y_test, label="Test", color="tomato")
axes[0].axvline(x=len(y_train), color="black", linestyle="--", linewidth=1, label="Split point")
axes[0].set_title("ERSSTv5 Nino3.4 SST Temperature (Z-scored)")
axes[0].set_xlabel("Time step (months)")
axes[0].set_ylabel("Z-scored SST Temperature")
axes[0].legend()

axes[1].hist(y_train, bins=30, alpha=0.6, label="Train", color="steelblue")
axes[1].hist(y_test, bins=20, alpha=0.6, label="Test", color="tomato")
axes[1].set_title("Distribution of Target")
axes[1].set_xlabel("Z-scored SST Temperature")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
plt.savefig(data_dir / "data_split_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Build the Model

In [ ]:
model = LinearRegression()

## 5. Train the Model

In [ ]:
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

## 6. Evaluate on the Test Set

In [ ]:
def evaluate_regression(y_true, y_pred, split_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"--- {split_name} ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R^2:  {r2:.4f}")

    return rmse, mae, r2

print("OLS Linear Regression Performance")
train_metrics = evaluate_regression(y_train, y_pred_train, "Train")
test_metrics = evaluate_regression(y_test, y_pred_test, "Test")

## 7. Visualize Results

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", key=abs, ascending=False)

print("OLS coefficients sorted by absolute value:")
print(coef_df.to_string(index=False))
print(f"Intercept: {model.intercept_:.4f}")

plt.figure(figsize=(10, 5))
colors = ["tomato" if c < 0 else "steelblue" for c in coef_df["Coefficient"]]
plt.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("OLS Coefficients for Lagged SST Features")
plt.xlabel("Coefficient Value")
plt.tight_layout()
plt.savefig(data_dir / "ols_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(y_test, label="Actual", color="steelblue", linewidth=1.5)
axes[0].plot(y_pred_test, label="OLS Predicted", color="tomato", linestyle="--", linewidth=1.5)
axes[0].set_title("OLS Predictions vs Actual (Test Set)")
axes[0].set_xlabel("Test Sample")
axes[0].set_ylabel("Scaled SST")
axes[0].legend()

plot_min = min(y_test.min(), y_pred_test.min())
plot_max = max(y_test.max(), y_pred_test.max())
axes[1].scatter(y_test, y_pred_test, alpha=0.5, color="steelblue", s=20)
axes[1].plot([plot_min, plot_max], [plot_min, plot_max], "r--", linewidth=1.5, label="Perfect fit")
axes[1].set_title(f"OLS Scatter, Test R^2 = {test_metrics[2]:.3f}")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")
axes[1].legend()

plt.tight_layout()
plt.savefig(data_dir / "ols_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save Test Predictions and Metrics

In [ ]:
predictions_df = pd.DataFrame({
    "y_actual": y_test,
    "y_pred_OLS": y_pred_test
})
predictions_df.to_csv(data_dir / "ols_test_predictions.csv", index=False)

metrics_df = pd.DataFrame({
    "Model": ["OLS Linear Regression"],
    "RMSE_Test": [test_metrics[0]],
    "MAE_Test": [test_metrics[1]],
    "R2_Test": [test_metrics[2]],
})
metrics_df.to_csv(data_dir / "ols_metrics.csv", index=False)

display(metrics_df)
print("Saved predictions to:", (data_dir / "ols_test_predictions.csv").resolve())
print("Saved metrics to:", (data_dir / "ols_metrics.csv").resolve())